In [10]:
import json
import re

In [11]:
def get_answer_text(answer_letter, options):
    """
    answer_letter: 'A'/'B'/'C'/'D'
    options: ["A. xxx", "B. yyy", ...]
    return: 'xxx' or 'yyy' (与answer_letter匹配的文本内容)
    """
    for opt in options:
        m = re.match(r"([A-D])\.\s*(.*)", opt)
        if m:
            if m.group(1) == answer_letter:
                return m.group(2).strip()
    return answer_letter  # fallback: in case no match found

In [12]:
with open('questions_omni.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [13]:
vid2qa = {}
for entry in data:
    video_path = entry['video_path']
    if video_path not in vid2qa:
        vid2qa[video_path] = {
            "video_id": video_path.split('/')[-1].split('.')[0],
            "video_path": video_path,
            "duration": 0,  # 没有duration，可以后续完善
            "conversations": []
        }
    for qa in entry["questions"]:
        # 查找 answer 对应的真实文本
        answer_letter = qa["answer"]  # e.g. "B"
        options = qa.get("options", [])
        answer_text = get_answer_text(answer_letter, options)
        # 处理 choices 数组（去掉"A. "/"B. "前缀，仅保文本，可选）
        # 如果你需要保留 "A. "，请用 options；如果只要文本，就如下处理
        choices_text = []
        for opt in options:
            m = re.match(r"([A-D])\.\s*(.*)", opt)
            if m:
                choices_text.append(m.group(2).strip())
            else:
                choices_text.append(opt)
        vid2qa[video_path]["conversations"].append({
            "question": qa["question"],
            "choices": choices_text,
            "answer": answer_text,
            "question_type": qa.get("task_type", "")
        })
output = list(vid2qa.values())

In [14]:
with open('question_omni_rekv.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=4)